# 學習測試(基礎架構).ipynb

步態畢業專題研究筆記。此檔保留探索步驟，已清除輸出、執行資訊及 Colab 中繼資料。

包含獨立實驗與尚未統一的座標定義，不保證由上而下可執行；使用整合流程請從根目錄 README 開始。舊輸出不作為本次驗證結果。


In [ ]:
import os
import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files
import io
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import time


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用設備：{device}")


In [ ]:
uploaded_files = files.upload()

all_data = pd.DataFrame()


In [ ]:
def step_x(x1, x2):
    return abs(x2 - x1)


In [ ]:
merged_count = 0
skipped_count = 0
total_files = len(uploaded_files)
all_data = pd.DataFrame()

def calculate_angle_change(x1, y1, x2, y2, x3, y3):
    a_x = x2 - x1
    a_y = y2 - y1
    b_x = x3 - x2
    b_y = y3 - y2
    dot_product = a_x * b_x + a_y * b_y
    magnitude_a = math.hypot(a_x, a_y)
    magnitude_b = math.hypot(b_x, b_y)
    if magnitude_a * magnitude_b == 0:
        return 0
    cos_angle = dot_product / (magnitude_a * magnitude_b)
    cos_angle = max(-1, min(1, cos_angle))
    return math.acos(cos_angle) * (180 / math.pi)

for file_name in uploaded_files.keys():
    try:
        df = pd.read_csv(io.BytesIO(uploaded_files[file_name]))

        if 'kpts' in df.columns:
            df['kpts'] = df['kpts'].str.replace('"', '').str.strip()

            kpts_columns = df['kpts'].str.split(',', expand=True)
            for i in range(kpts_columns.shape[1]):
                kpts_columns.rename(columns={i: f'kpt_{i+1}'}, inplace=True)
            df = df.join(kpts_columns, rsuffix='_kpts')

            for col in kpts_columns.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)


            if 'kpt_46' in df.columns and 'kpt_49' in df.columns:
                x16 = df['kpt_46']
                x17 = df['kpt_49']
                step_lengths = [abs(x16.iloc[i] - x17.iloc[i]) for i in range(len(x16))]
                average_step_length = sum(step_lengths) / len(step_lengths) if step_lengths else 0

                print(f"{file_name} 步距列表（部分）：{step_lengths[:5]}")
                print(f"{file_name} 平均步距：{average_step_length}")


                x11 = df['kpt_31']
                y11 = df['kpt_32']
                x12 = df['kpt_34']
                y12 = df['kpt_35']
                x14 = df['kpt_40']
                y14 = df['kpt_41']
                x15 = df['kpt_44']
                y15 = df['kpt_45']
                x18 = (x11 + x12) / 2
                y18 = (y11 + y12) / 2

                angle_changes = [
                    calculate_angle_change(x14.iloc[i], y14.iloc[i], x18.iloc[i], y18.iloc[i], x15.iloc[i], y15.iloc[i])
                    for i in range(len(x18))
                ]

                print(f"{file_name} 角度變化列表（部分）：{angle_changes[:5]}")

                df['step_length'] = step_lengths
                df['angle_change'] = angle_changes


                if df.columns.duplicated().any():
                    print(f"文件 {file_name} 存在重複列名，正在處理...")
                    df = df.loc[:, ~df.columns.duplicated()]


                df = df.reset_index(drop=True)
                all_data = all_data.reset_index(drop=True)


                all_data = pd.concat([all_data, df], ignore_index=True, axis=0)
                merged_count += 1
            else:
                print(f"文件 {file_name} 中缺少必要的關節點數據，跳過該文件。")
                skipped_count += 1
        else:
            print(f"文件 {file_name} 中沒有 'kpts' 列，跳過該文件。")
            skipped_count += 1

    except pd.errors.ParserError:
        print(f"文件 {file_name} 解析失敗，可能是格式錯誤，已跳過。")
        skipped_count += 1


In [ ]:
print(f"已合併的檔案量：{merged_count}")
print(f"未合併的檔案量：{skipped_count}")
print(f"總檔案量：{total_files}")


進行監督式學習

In [ ]:
labels = np.random.randint(0, 4, len(all_data))


In [ ]:
features = all_data[['step_length', 'angle_change']].values


In [ ]:
scaler = StandardScaler()
features = scaler.fit_transform(features)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(features, labels, test_size=0.2, random_state=42, stratify=labels)


In [ ]:
class PatientDataset(Dataset):
    def __init__(self, data, labels):
        self.features = data
        self.labels = labels

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        x = self.features[idx]
        y = self.labels[idx]
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.long)


In [ ]:
train_dataset = PatientDataset(X_train, y_train)
test_dataset = PatientDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


In [ ]:
class SeverityClassifier(nn.Module):
    def __init__(self, input_size, num_classes):
        super(SeverityClassifier, self).__init__()
        self.fc1 = nn.Linear(input_size, 64)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(64, 32)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(32, num_classes)

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu1(out)
        out = self.fc2(out)
        out = self.relu2(out)
        out = self.fc3(out)
        return out


In [ ]:
model = SeverityClassifier(input_size=2, num_classes=4).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)


In [ ]:
def train(num_epochs):
    model.train()
    for epoch in range(num_epochs):
        for i, (features, labels) in enumerate(train_loader):
            features = features.to(device)
            labels = labels.to(device)

            outputs = model(features)
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()


        scheduler.step()


        if (epoch+1) % 10 == 0:
            current_lr = scheduler.get_last_lr()[0]
            print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}, Learning Rate: {current_lr:.6f}")


In [ ]:
train(num_epochs=100)


In [ ]:
def test():
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for features, labels in test_loader:
            features = features.to(device)
            labels = labels.to(device)
            outputs = model(features)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    print(f"模型在測試集上的準確率為：{100 * correct / total:.2f}%")


In [ ]:
test()


In [ ]:
start_time = time.time()
test()
end_time = time.time()
print(f"模型評估時間：{end_time - start_time:.2f} 秒")
